<a href="https://colab.research.google.com/github/E-tech-coder/DataScienceCapstoneProject/blob/Elena/TFIDF-CosineSimilarity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/E-tech-coder/DataScienceCapstoneProject.git

Cloning into 'DataScienceCapstoneProject'...
remote: Enumerating objects: 213, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 213 (delta 88), reused 20 (delta 20), pack-reused 88 (from 1)
Receiving objects: 100% (213/213), 1.67 MiB | 8.99 MiB/s, done.
Resolving deltas: 100% (121/121), done.


# 1. Data Loading and Preparation

In [4]:
import pandas as pd
df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv").set_index("person_id").reset_index()

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2615 entries, 0 to 2614
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   person_id           2615 non-null   int64  
 1   organization        2615 non-null   object 
 2   position            2615 non-null   object 
 3   startDate           2615 non-null   object 
 4   endDate             2615 non-null   object 
 5   status              2615 non-null   object 
 6   department          2615 non-null   object 
 7   seniority           2615 non-null   object 
 8   job_count           2615 non-null   int64  
 9   job_duration_years  2615 non-null   float64
dtypes: float64(1), int64(2), object(7)
memory usage: 204.4+ KB


# 2.Department Dataset

In [7]:
dpt = pd.read_csv("/content/DataScienceCapstoneProject/department-v2.csv")

In [9]:
dpt['label'].drop_duplicates()

,label
0,Marketing
1,Project Management
171,Administrative
202,Business Development
374,Consulting
476,Human Resources
487,Information Technology
2609,Other
2613,Purchasing
2636,Sales


In [10]:
dpt

,text,label
0,Adjoint directeur communication,Marketing
1,Advisor Strategy and Projects,Project Management
2,Beratung & Projekte,Project Management
3,Beratung & Projektmanagement,Project Management
4,Beratung und Projektmanagement kommunale Partner,Project Management
...,...,...
10140,VP Sales D.A.CH.,Sales
10141,VP Sales DACH,Sales
10142,VP Sales Development & Strategy,Sales
10143,VP Sales Germany,Sales


Explanation

This dataset contains department job position texts and department names used as reference documents.

# 3. Simple Bag of Words(BOW)

## 3.1 Constructing Bag of Words per Department

In [11]:
import re

BOW = dict()

for label in dpt['label'].drop_duplicates() :
  x = []
  for text in dpt[dpt["label"] == label]["text"]:
    tokens = re.split(r"[ ./&-()|-]", text.lower().strip())
    tokens = [t for t in tokens if t]
    x.extend(tokens)
  BOW[label] = list(set(x))


Explanation

For each department label:

*   Tokenize all related text descriptions
*   Convert text to lowercase
*   Split on punctuation and separators
*   Remove empty tokens
*   Store unique words per department
*   Result:BOW[label] = vocabulary representing that department


This creates a very simple keyword-based representation for each department.

# 4. Job Title Tokenization

In [ ]:
import re

def split_position(text):
  # Lowercase and split on space, /, &, -
  return re.split(r'[ /&-]+',text.lower().strip())

df["position_words"] = df["position"].apply(split_position)

Explanation

Job titles are:
*   Lowercased
*   Split into individual words

This prepares job titles for word matching.

# 5. Department Prediction Using BoW

In [ ]:
df["WordAppearance"] = 0
df["PredictedPosition_BOW"] =""

In [ ]:
for label, BOW_words in BOW.items():
  bow_set = set(BOW_words)
  # Count how many words of the position appear in the bag of words of the label
  for idx,words in df["position_words"].items():
    count=sum(word in bow_set for word in words)
        # Update prediction if the current word count is greater than the previous one
    if count > df.at[idx, "WordAppearance"] :
      df.at[idx, "WordAppearance"] = count
      df.at[idx, "PredictedPosition_BOW"] = label

Explanation

For each job title:


*   Count how many words appear in each department’s Bag of Words
*   Assign the department with the highest word overlap

This is a rule-based, frequency-only classifier
No weighting, no context, no semantics.

#6. TF-IDF with Sklearn (Department)

## 6.1 Why TF-IDF
BoW treats all words equally.
TF-IDF gives higher importance to discriminative words and down-weights common ones.

## 6.2 Building TF-IDF Vectors

In [ ]:
corpus = []
labels = []

for key, value in BOW.items():
  labels.append(key)
  corpus.append(value)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# turn seperated words into a string sentence in each corpus
corpus = [" ".join(doc) for doc in corpus]
titles = labels



*   Each department becomes one document
*   Words are joined into a sentence-like string



In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1,1), lowercase=True)
vector = vectorizer.fit_transform(corpus)

Explanation
*   Learn TF-IDF weights for all department vocabularies
* Output shape:  
 (number of departments, number of unique words)


In [ ]:
tfidf_df = pd.DataFrame(vector.toarray(), index = titles, columns = vectorizer.get_feature_names_out())

In [ ]:
tfidf_df.shape

(11, 3405)

## 6.3 TF-IDF Score Matching (Manual)

In [ ]:
df["Tfidf_score"] = 0
df["tfidf_predict"] = " "

In [ ]:
for idx, words in df["position_words"].items():
  for label in tfidf_df.index :
    score = 0
    for word in words :
      if word in list(tfidf_df.columns) and (tfidf_df.at[label,word] != 0) :
        score = score + tfidf_df.at[label,word]
    if score > df.at[idx, "Tfidf_score"]:
      df.at[idx, "Tfidf_score"] = score
      df.at[idx, "tfidf_predict"] = label


/tmp/ipython-input-2041765854.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.013857517247410037' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[idx, "Tfidf_score"] = score


Explanation

For each job title:
* Sum TF-IDF weights of overlapping words per department
* Assign department with highest TF-IDF score

This improves over BoW by weighting important words more

# 7. Cosine Similarity (Vector-Based Matching)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

X_positions = vectorizer.transform(df['position'])
similarity = cosine_similarity(X_positions, tfidf_df)

Explanation

* Transform job titles into TF-IDF vectors
* Compute cosine similarity between:
  * Job title vector
  * Department vectors

* Cosine similarity measures angle, not magnitude: Good for short text comparison.

In [ ]:
import numpy as np
df["cosine_predict"] = tfidf_df.index[similarity.argmax(axis = 1)]
df["cosine_ifidf_score"] = similarity.max(axis = 1)

In [ ]:
df.to_csv("/content/DataScienceCapstoneProject/Cosine-Position-Predict.csv",sep = ";")

# 8. Model Evaluation

In [ ]:
accuracy = {}

accuracy["BOW_accuracy"] = round(len(df[df['department']==df["PredictedPosition_BOW"]])/len(df),5)
accuracy["TFIDF_accuracy"] = round(len(df[df['department']==df["tfidf_predict"]])/len(df),5)
accuracy["CosineSimilarity_accuracy"] = round(len(df[df['department']==df["cosine_predict"]])/len(df),5)

In [ ]:
accuracy

{'BOW_accuracy': 0.06861,
 'TFIDF_accuracy': 0.23806,
 'CosineSimilarity_accuracy': 0.24375}

All methods show low accuracy.

The accuracies using TFIDF and cosine similarity is low .
Reasons:
1. Job titles are short. Short texts = sparse vectors. TF-IDF often gives very small overlaps. Cosine similarity struggles if there’s little shared vocabulary

2. TF-IDF doesn’t capture synonyms. “Project Manager” vs “Program Manager” → different words → low similarity
“Data Analyst” vs “Business Analyst” → low overlap
TF-IDF only sees exact tokens.
3. Label descriptions may be noisy or inconsistent.
If label TF-IDF vectors come from long documents, words like “the”, “department” dilute importance. Short position titles match poorly.

# 9. Why Accuracy Is Low

Key Reasons

1. Short job titles
  * Sparse vectors
  * Few overlapping words
2. No synonym handling
  * “Data Analyst” vs “Business Analyst”
  * “Project Manager” vs “Program Manager”
3. No semantic understanding
  * TF-IDF only matches exact tokens
4. Label text mismatch
  * Department descriptions are longer and noisier than job titles

These methods are baseline techniques, not production-grade classifiers.

# Seniority Prediction Section

## 10. Date Cleaning and Work Experience Calculation

In [ ]:
df.loc[(df["endDate"].isna()) & (df["status"]=="ACTIVE"), "endDate"] = datetime.today().strftime('%Y-%m')
df["WorkYears"] = round((df["endDate"] - df["startDate"]).dt.days / 365, 3)

Explanation

* Active jobs with missing end dates are assumed to end today
* This allows work duration to be calculated

## 11. Seniority Bag of Words



In [ ]:
seniority_df = pd.read_csv('/content/DataScienceCapstoneProject/seniority-v2.csv')

In [ ]:
import re
BOW_seniority = {}

for label in set(seniority_df["label"].tolist()):
  x = []
  for text in seniority_df[seniority_df["label"]==label]["text"]:
    tokens = re.split(r"[ ./&()|-]", text.lower().strip())
    tokens = [t for t in tokens if t]
    x.extend(tokens)
  BOW_seniority[label] = list(x)

In [ ]:
titles = []
corpus = []
for title, cor in BOW_seniority.items():
  titles.append(title)
  corpus.append(cor)

# In the "seniority.csv" file, there's no title called "professional". But in the linkedin CV data, there's a seniority title called "professional".
# So I created the Bag of words for "Professional" using the CV data.
BOW_professional = [" ".join(word for word in df[df["seniority"]=="Professional"]["position"].str.lower().tolist())]
BOW_seniority["Professional"] = BOW_professional

## 12. TF-IDF + Cosine Similarity for Seniority

In [ ]:
from numpy import vectorize
# Cosine similarity

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
# turn seperated words into a string sentence in each corpus

corpus = [" ".join(doc) for doc in corpus]

vectorizer = TfidfVectorizer(ngram_range=(1,3))
vector = vectorizer.fit_transform(corpus)
tfidf_df = pd.DataFrame(vector.toarray(), index = titles, columns = vectorizer.get_feature_names_out())


* Predict seniority based purely on job title semantics

A person can have 10 years of experience. But still start as Junior in a new field

Therefore,
Position title is more reliable than total years.
Work years are used as supporting context, not the main predictor.

In [ ]:
import numpy as np
X_positions = vectorizer.transform(df["position"])
similarity = cosine_similarity(X_positions, tfidf_df)
df["TFIDF-seniority"]=tfidf_df.index[similarity.argmax(axis=1)]

In [ ]:
# Calculate accuracy using only text analysis without considering work years
accuracy = len(df[df["seniority"]==df["TFIDF-seniority"]])/len(df)
accuracy

This part of work intentionally explores baseline NLP techniques such as Bag of Words, TF-IDF, and cosine similarity to demonstrate their strengths and limitations when applied to short, real-world job titles. The low accuracy highlights why more advanced approaches such as word embeddings, semantic similarity, or supervised learning would be required for production-level performance.